# Single-cell WFCM: two-stage selection and parametric bootstrap


Stage 1 fixes m=2, fits k=2,…,7, and minimizes the manuscript’s normalized weighted XBI, k*sum(w_j*u_ij**m*distance**2)/(n*minimum squared center separation). Reference partitions must have at least five maximum-membership assignments per cluster. This application safeguard was introduced after diagnosing an artificial minimum from a one-cell, near-boundary k=7 fit; it is not part of the original Algorithm 1. All raw scores and exclusion reasons are saved, and sensitivity to minimum sizes 2,5,10 is reported. The code computes the eligible minimum; k=3 is not hardcoded as the outcome.

Stage 2 fixes the selected k and fits each m in a ten-candidate grid np.geomspace(1.1,6,10) spanning 1.1–6, selecting the smallest refined numerical NLL. Both stages share one fixed, data-adaptive integration cloud. The proposal combines a data GMM, isotropic Gaussian coverage at its centers and a broad Student-t component. Integration points are conditionally iid, as described in the manuscript, and use the full mixture density in their denominators. The data, original initialization seed=321/default_rng, cohort seed=42 and update formulas are preserved. An independent integration cloud checks the m ranking before inference; finite-cloud approximation error is reported rather than assumed absent.

Bootstrap observations are independent draws from the fitted density. Every refit rebuilds its proposal/cloud at fixed selected k,m and aligns centers and weights. Scalar SDs use B; covariance estimates use B-1; center/weight region formulas and simplex restriction follow the supplied algorithm. Neither stage of selection nor preprocessing/PCA is repeated by bootstrap, so intervals do not adjust for selection uncertainty. Numerical termination and diagnostics do not prove a global optimum, true-k recovery or interval coverage.

The 3,000-cell cohort is stratified using the supplied reference cell types, and fitting uses two standardized PCs as before. Cell-type names are applied only after fitting for descriptive annotation. Run from this folder and configure DATA_PATH if needed. The original initialization is sampled rows with sigma=1; this choice should be stated in the numerical-implementation paragraph, which currently describes a different initialization.

The k-selection curve is always plotted. An optional full weighted-XBI grid is available in the last cell. It uses a new criterion-specific cache; the previous ordinary-XB grid is historical and is not used to select k or m.

### Required data and reference result

Data-preparation source: https://github.com/lucylgao/clusterpval-experiments/blob/master/real-data-code/zheng.R . The processed data must be obtained/generated separately; it is not silently downloaded or substituted.


## Setup
This is **one self-contained code file**: all WFCM fitting, integration, inference and plotting functions are embedded. **Datasets are separate inputs**, not embedded in this notebook. Start a fresh Python kernel, configure the data path, then run every cell in order.

Tested Python: 3.12.9. A matching environment can be installed with:
```python
%pip install numpy==2.5.4 scipy==1.18.1 pandas==3.1.0 scikit-learn==1.9.1 matplotlib==3.11.2 seaborn==0.13.2 pyreadr==0.5.9
```
Run the installation command separately if needed; this notebook does not automatically modify your environment. Full inference uses B=100 and can be slow. Decreasing B or the integration budget is a diagnostic run, not reproduction of the reference result.


In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
os.environ.setdefault('OMP_MAX_ACTIVE_LEVELS','1')
os.environ.setdefault('LOKY_MAX_CPU_COUNT','1')


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OMP_MAX_ACTIVE_LEVELS"] = "1"


In [ ]:
from pathlib import Path
import json
import pyreadr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
# Portable data lookup; set WFCM_DATA_DIR or edit the input setting below.
import os, hashlib, platform
from pathlib import Path

def locate_data(filename, required=True):
    preferred=Path(os.environ.get('WFCM_DATA_DIR','data'))/filename
    candidates=[preferred,Path.cwd()/'data'/filename,Path.cwd().parent/'data'/filename,Path.cwd()/filename]
    for candidate in candidates:
        if candidate.is_file():return candidate.resolve()
    if required:
        raise FileNotFoundError(f'Required dataset {filename!r} is not bundled. Put it in data/, set WFCM_DATA_DIR, or edit the input path. No substitute data will be used.')
    return preferred.resolve()

# Save every displayed figure, including plots without an explicit savefig call.
# Individual records are written only to local output files after execution.
import matplotlib.pyplot as plt
_base_show=getattr(plt.show,'_wfcm_base_show',plt.show)
_saved_figure_files=[]
def _show_and_save(*args,**kwargs):
    directory=Path(OUTPUT_DIR)/'figures';directory.mkdir(parents=True,exist_ok=True)
    for number in plt.get_fignums():
        figure=plt.figure(number)
        if not getattr(figure,'_wfcm_exported',False):
            file=directory/f'figure_{len(_saved_figure_files)+1:02d}.png'
            figure.savefig(file,dpi=180,bbox_inches='tight');figure._wfcm_exported=True
            _saved_figure_files.append(str(file))
    return _base_show(*args,**kwargs)
_show_and_save._wfcm_base_show=_base_show
plt.show=_show_and_save


## Single-Cell RNA Sequencing Data (Zheng et al. 2017)

Single-cell RNA sequencing (scRNA-seq) data captures gene expression profiles at the resolution of individual cells, enabling researchers to explore cellular heterogeneity. A common analysis strategy involves clustering cells to infer putative cell types, followed by differential gene expression analysis between the resulting groups. However, conventional methods often overlook the uncertainty inherent in the clustering step (Luecken and Theis, 2019; Lähnemann et al., 2020; Deconinck et al., 2021) [1]. In our analysis, we utilize a publicly available scRNA-seq dataset from peripheral blood mononuclear cells (PBMCs), originally pre-classified prior to sequencing by Zheng et al. (2017), to illustrate WFCM fitting and fixed-(k,m) parametric bootstrap uncertainty for centers and weights.

### Data Pre-processing

For our analysis, we focus on a subset of peripheral blood mononuclear cells consisting of memory T cells, B cells, and monocytes. Following standard single-cell RNA-seq preprocessing procedures (Duò, Robinson, and Soneson, 2018), we filter out cells with a high proportion of mitochondrial gene expression, cells with an unusually low or high number of expressed genes, and those with low total UMI counts. To account for size differences, we normalize the data such that each cell has the same total count equal to the average across all cells. We then apply a log2 transformation with a pseudocount of 1. To reduce dimensionality while retaining informative features, we select the top 500 genes with the highest average expression prior to normalization. This preprocessing pipeline is applied to the combined set of memory T cells, B cells, and monocytes. We construct an imbalanced clustered dataset, consisting of 1,500 memory T cells, 1,000 B cells, and 500 monocytes, totaling 3,000 cells. The dataset allows us to assess the robustness of our method under imbalanced cluster structures.


### Reference
[1] Gao, Lucy L., Jacob Bien, and Daniela Witten. "Selective inference for hierarchical clustering." Journal of the American Statistical Association 119.545 (2024): 332-342.

In [ ]:
# Run from the new notebook folder; paths are resolved and printed for clarity.
DATA_PATH = locate_data('processed_mix3_df.rds')
OUTPUT_DIR = Path('results_cell').resolve()
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Configure DATA_PATH to the processed single-cell RDS: {DATA_PATH}')
print('Input:', DATA_PATH)
print('Outputs:', OUTPUT_DIR)
result = pyreadr.read_r(str(DATA_PATH))
df = result[None]
X = df.drop(columns=['CellType']).values
labels = df['CellType'].values
tcell_id = np.where(labels == 'Memory T cell')[0]
bcell_id = np.where(labels == 'B cell')[0]
mono_id = np.where(labels == 'Monocyte')[0]

In [ ]:
# Reproducible cohort sampling.
np.random.seed(42)
# Sample 500, 1000, 1500 cells from each type
ss_id2 = np.concatenate([
    np.random.choice(tcell_id, 1500, replace=False),
    np.random.choice(bcell_id, 1000, replace=False),
    np.random.choice(mono_id, 500, replace=False)
])
X2_sampled = X[ss_id2, :]  


# Ensure that X1 is a dense matrix (if it's sparse)
X2_dense = X2_sampled.toarray() if isinstance(X2_sampled, np.ndarray) is False else X2_sampled


# PCA: We reduce to 2 principal components
pca = PCA(n_components=2, random_state=42)
pca_data2 = pca.fit_transform(X2_dense)

# Create a DataFrame for the PCA results with PC1, PC2, and CellType (from processed_mix3)
pca_df2 = pd.DataFrame(pca_data2, columns=['PC1', 'PC2'])
pca_df2['CellType'] = labels[ss_id2]

# Plot PCA results
plt.figure(figsize=(10, 6))
sns.scatterplot(data=pca_df2, x='PC1', y='PC2', hue='CellType', palette='Dark2', s=100)
plt.title('First two PCs of Single-Cell RNA Sequencing Data', fontsize = 18, weight = "bold")
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(title='Cell types', loc='upper right', ncol=1)
#plt.gca().set_aspect(6/6)  # Adjust the aspect ratio
plt.show()


# scale the data
scaler = StandardScaler()
pc_scaled2 = scaler.fit_transform(pca_df2[['PC1', 'PC2']])
df_scaled2 = pd.DataFrame(pc_scaled2, columns=pca_df2[['PC1', 'PC2']].columns)
df_combined2 = pd.concat([df_scaled2, pca_df2[['CellType']]], axis=1)




In [ ]:
# Check variance explained (for rebuttal)
pca_check = PCA(n_components=min(20, *X2_dense.shape), random_state=42)
pca_check.fit(X2_dense)
cumvar = np.cumsum(pca_check.explained_variance_ratio_)
print("Explained variance ratio, PC1–PC5:", pca_check.explained_variance_ratio_[:5].round(4))
print("Cumulative variance, PC1–PC5:", cumvar[:5].round(4))

In [ ]:
"""WFCM fitting and fixed-(k,m) parametric bootstrap from the manuscript.

Dependencies: numpy, scipy, scikit-learn. No data preprocessing is performed.
All likelihoods use f(x) proportional to exp(-h(x)/sigma**2), where
h(x) = [sum_j (w_j ||x-v_j||**2)**(-1/(m-1))]**(-(m-1)).
"""
import numpy as np
from scipy.optimize import minimize, linear_sum_assignment
from scipy.special import logsumexp
from scipy.spatial.distance import cdist
from scipy.stats import chi2, norm
from sklearn.mixture import GaussianMixture


def _data(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or min(X.shape) < 1 or not np.isfinite(X).all():
        raise ValueError('Expected a finite, nonempty samples-by-features matrix.')
    return X


from scipy.stats import multivariate_t

class DefensiveProposal:
    """Data GMM + isotropic coverage at its centers + heavy-tailed defense.

    The isotropic component prevents thin data-covariance directions from
    undersampling the model's isotropic Gaussian tails. All densities are
    evaluated as the complete mixture, including the Student-t component.
    """
    def __init__(self, X, n_components, seed):
        self.seed = int(seed)
        self.gmm = GaussianMixture(n_components=n_components, covariance_type='full',
                                   random_state=self.seed).fit(X)
        if not self.gmm.converged_:
            raise RuntimeError('Importance-proposal GMM did not converge.')
        self.loc = X.mean(axis=0)
        self.d = X.shape[1]
        self.variance = max(float(np.var(X,axis=0,ddof=1).mean()), 1e-8)
        self.shape = 4.*self.variance*np.eye(self.d)
        self.df = 5.
        J = len(self.gmm.weights_)
        self.weights = np.r_[.2*self.gmm.weights_, np.full(J,.7/J), .1]
        self.name = '0.2 data GMM + 0.7 isotropic center mixture + 0.1 Student-t(df=5); iid Monte Carlo'
    def sample(self, n):
        if int(n) != n or n < 2:
            raise ValueError('At least two integer integration samples are required.')
        n = int(n)
        U = np.random.default_rng(self.seed).random((n, self.d+2))
        U = np.clip(U, np.finfo(float).eps, 1-np.finfo(float).eps)
        labels = np.minimum(np.searchsorted(np.cumsum(self.weights), U[:,0], side='right'), len(self.weights)-1)
        G = norm.ppf(U[:,1:self.d+1]);Z = np.empty((n,self.d));J=len(self.gmm.weights_)
        for j in range(J):
            use = labels == j
            Z[use] = self.gmm.means_[j] + G[use] @ np.linalg.cholesky(self.gmm.covariances_[j]).T
            use = labels == J+j
            Z[use] = self.gmm.means_[j] + G[use]*np.sqrt(self.variance)
        use = labels == 2*J
        radial = np.sqrt(self.df / chi2.ppf(U[use,-1], self.df))
        Z[use] = self.loc + G[use]*(2*np.sqrt(self.variance))*radial[:,None]
        return Z, labels
    def score_samples(self, Z):
        d2=((Z[:,None,:]-self.gmm.means_[None,:,:])**2).sum(axis=2)
        log_iso=logsumexp(-.5*d2/self.variance,axis=1)-np.log(len(self.gmm.weights_))-.5*self.d*np.log(2*np.pi*self.variance)
        log_t=multivariate_t.logpdf(Z,loc=self.loc,shape=self.shape,df=self.df)
        return logsumexp(np.vstack([np.log(.2)+self.gmm.score_samples(Z),
                                   np.log(.7)+log_iso,np.log(.1)+log_t]),axis=0)

def build_proposal(X, num_samples=20000, n_components=None, seed=42):
    X = np.asarray(X,dtype=float)
    if X.ndim != 2 or X.shape[0]<2 or X.shape[1]<1 or not np.isfinite(X).all():
        raise ValueError('Proposal requires finite data with at least two rows and one feature.')
    components = min(3,len(X)) if n_components is None else n_components
    proposal = DefensiveProposal(X,components,seed)
    Z,_ = proposal.sample(num_samples)
    return Z,proposal.score_samples(Z),proposal

def _unpack(theta, k, d):
    centers = theta[1:1+k*d].reshape(k, d)
    logits = np.r_[theta[1+k*d:], 0.0]
    return theta[0], centers, logits-logsumexp(logits)


def _pack(sigma, centers, w):
    w = np.asarray(w, dtype=float)
    if not np.isfinite(sigma) or sigma <= 0 or np.any(w <= 0) or not np.isfinite(w).all():
        raise ValueError('The density requires positive scale and strictly positive weights.')
    w = w/w.sum()
    return np.r_[np.log(sigma), np.asarray(centers).ravel(), np.log(w[:-1])-np.log(w[-1])]


def _terms(X, centers, logw, m):
    # Use log space; only distances at floating-point zero receive a floor.
    d2 = np.maximum(cdist(X, centers, 'sqeuclidean'), 1e-200)
    scores = -(np.log(d2)+logw)/(m-1)
    lse = logsumexp(scores, axis=1)
    return np.exp(scores-lse[:, None]), np.exp(-(m-1)*lse), d2


def memberships(X, centers, w, m):
    if m <= 1:
        raise ValueError('m must exceed 1.')
    return _terms(_data(X), np.asarray(centers), np.log(w), m)[0]


def nll_gradient(theta, X, Z, logq, m, k, compute_center_gradient=True):
    """IS NLL and analytic gradient in log-scale / center / weight-logit coordinates."""
    logsigma, centers, logw = _unpack(theta, k, X.shape[1])
    w = np.exp(logw)
    with np.errstate(over='ignore', invalid='ignore', divide='ignore'):
        ux, hx, dx = _terms(X, centers, logw, m)
        uz, hz, dz = _terms(Z, centers, logw, m)
        ex, ez = hx*np.exp(-2*logsigma), hz*np.exp(-2*logsigma)
        lr = -ez-logq
        lse = logsumexp(lr)
        prob = np.exp(lr-lse)
        value = len(X)*(lse-np.log(len(Z))) + ex.sum()

        def gradient(A, U, e, d2, a):
            t = a[:, None]*e[:, None]*U/d2
            # Accumulate displacements directly: subtracting two large matrix
            # products loses precision when a center nearly equals a data point.
            if compute_center_gradient:
                displacement = centers[None, :, :]-A[:, None, :]
                gv = 2*np.einsum('ij,ijd->jd', t, displacement)
            else:
                gv = np.zeros_like(centers)
            gw = ((a*e)[:, None]*(U-w)).sum(axis=0)[:-1]
            return np.r_[-2*np.dot(a, e), gv.ravel(), gw]

        grad = gradient(X, ux, ex, dx, np.ones(len(X)))-gradient(Z, uz, ez, dz, len(X)*prob)
    if not np.isfinite(value) or not np.isfinite(grad).all():
        return np.inf, np.zeros_like(theta)
    return float(value), grad


def importance_diagnostics(fit, Z, logq):
    _, h, _ = _terms(Z, fit['centers'], np.log(fit['w']), fit['m'])
    lr = -h/fit['sigma']**2-logq
    lse = logsumexp(lr)
    p = np.exp(lr-lse)
    return dict(logZ=float(lse-np.log(len(Z))), ESS=float(1/np.sum(p*p)),
                ESS_fraction=float(1/(len(Z)*np.sum(p*p))), largest_IS_weight=float(p.max()))


def fit_fixed(X, Z, logq, m, k, initial_sigma=1., initial_centers=None,
              initial_w=None, seed=321, max_outer=200, epsilon=1e-5,
              delta=1e-5, inner_maxiter=1000, K_post=1000):
    """Algorithm emfuzzy for one m: block updates, then joint refinement.

    max_outer is an explicit operational guard; hitting it is reported as
    nonconvergence, never silently treated as satisfying the stopping rule.
    The weight softmax enforces the simplex throughout optimization.
    """
    X, Z = _data(X), _data(Z)
    logq = np.asarray(logq, dtype=float)
    if (m <= 1 or not np.isfinite(m) or not 1 <= k <= len(X)
            or X.shape[1] != Z.shape[1] or logq.shape != (len(Z),)
            or not np.isfinite(logq).all() or max_outer < 1 or inner_maxiter < 1
            or K_post < 0 or epsilon < 0 or delta < 0):
        raise ValueError('Invalid model, proposal, or optimizer configuration.')
    d = X.shape[1]
    centers = (X[np.random.default_rng(seed).choice(len(X), k, replace=False)].copy()
               if initial_centers is None else np.array(initial_centers, dtype=float, copy=True))
    if centers.shape != (k, d):
        raise ValueError('Initial centers must have shape (k,d).')
    w = np.full(k, 1/k) if initial_w is None else np.asarray(initial_w, dtype=float)
    if w.shape != (k,):
        raise ValueError('Initial weights must have shape (k,).')
    theta = _pack(initial_sigma, centers, w)
    value, _ = nll_gradient(theta, X, Z, logq, m, k)
    if not np.isfinite(value):
        raise RuntimeError('Initial likelihood is not finite.')
    sw_indices = np.r_[0, np.arange(1+k*d, len(theta))]
    history, converged, failures = [], False, 0

    def physical(t):
        ls, v, lw = _unpack(t, k, d)
        return np.r_[np.exp(ls), v.ravel(), np.exp(lw)]

    for iteration in range(1, max_outer+1):
        old_theta, old_value = theta.copy(), value
        _, v, lw = _unpack(theta, k, d)
        U, _, _ = _terms(X, v, lw, m)
        # The centroid update is unconditional, exactly as in the manuscript.
        logum = m*np.log(np.maximum(U, np.finfo(float).tiny))
        um = np.exp(logum-logsumexp(logum, axis=0))
        theta[1:1+k*d] = (um.T@X).ravel()

        def block_objective(sw):
            candidate = theta.copy()
            candidate[sw_indices] = sw
            f, g = nll_gradient(candidate, X, Z, logq, m, k, compute_center_gradient=False)
            return f, g[sw_indices]

        opt = minimize(block_objective, theta[sw_indices], jac=True, method='L-BFGS-B',
                       options=dict(maxiter=inner_maxiter, ftol=1e-12, gtol=1e-6, maxls=100))
        if not opt.success:
            failures += 1
        if not np.isfinite(opt.fun):
            raise RuntimeError('Nonfinite scale/weight optimization.')
        theta[sw_indices] = opt.x
        value, _ = nll_gradient(theta, X, Z, logq, m, k)
        step = float(np.linalg.norm(physical(theta)-physical(old_theta)))
        change = abs(value-old_value)
        history.append(dict(stage='blockwise', iteration=iteration, NLL=value,
                            parameter_change=step, NLL_change=change,
                            optimizer_success=bool(opt.success), message=str(opt.message)))
        converged = bool(opt.success and (step <= epsilon or change <= delta))
        if converged:
            break

    post_success, post_message, post_iterations, post_status = True, 'K_post=0: refinement skipped', 0, 0
    post_attempts = 0
    # A fresh L-BFGS history can recover from an abnormal line search. Both
    # attempts share the manuscript's TOTAL K_post iteration budget.
    if K_post:
        post_success = False
        for attempt, line_search_budget in enumerate((100, 200), start=1):
            remaining = K_post-post_iterations
            if remaining <= 0:
                break
            opt = minimize(nll_gradient, theta, args=(X, Z, logq, m, k), jac=True,
                           method='L-BFGS-B', options=dict(maxiter=remaining,
                           ftol=1e-12, gtol=1e-6, maxls=line_search_budget))
            trial_value, trial_gradient = nll_gradient(opt.x, X, Z, logq, m, k)
            accepted = bool(np.isfinite(trial_value) and trial_value <= value+1e-7)
            if accepted:
                theta, value = opt.x.copy(), float(trial_value)
            post_iterations += int(opt.nit)
            post_attempts = attempt
            post_success = bool(opt.success and accepted)
            post_status = int(opt.status) if accepted else 2
            post_message = str(opt.message) if accepted else 'Refinement returned a nonfinite or worsening point.'
            history.append(dict(stage='joint_refinement', iteration=post_iterations,
                                attempt=attempt, attempt_iterations=int(opt.nit),
                                NLL=value, optimizer_success=post_success,
                                optimizer_status=post_status, message=post_message,
                                gradient_inf_norm=float(np.max(np.abs(trial_gradient))),
                                accepted=accepted, maxls=line_search_budget))
            if post_success or post_status == 1:
                break
    ls, centers, lw = _unpack(theta, k, d)
    result = dict(sigma=float(np.exp(ls)), centers=centers.copy(), w=np.exp(lw),
                  U=_terms(X, centers, lw, m)[0], m=float(m), k=int(k), NLL=float(value),
                  theta=theta.copy(), history=history, converged=converged,
                  optimizer_failures=failures, post_success=post_success,
                  post_message=post_message, post_iterations=post_iterations,
                  post_status=post_status, post_attempts=post_attempts,
                  seed=int(seed), n_iter=iteration, engine='manuscript',
                  fit_options=dict(max_outer=max_outer, epsilon=epsilon, delta=delta,
                                   inner_maxiter=inner_maxiter, K_post=K_post))
    result['IS_diagnostics'] = importance_diagnostics(result, Z, logq)
    return result


def select_m(X, k, m_grid, num_is=20000, proposal_components=None,
             proposal_seed=674, seeds=(321,), **fit_options):
    """Fixed k; fit each m, refine, and select the smallest NLL (no XB filter)."""
    X = _data(X)
    grid = np.asarray(m_grid, dtype=float)
    if grid.ndim != 1 or not len(grid) or not np.isfinite(grid).all() or np.any(grid <= 1):
        raise ValueError('m_grid must contain finite values greater than one.')
    if not len(seeds):
        raise ValueError('Provide at least one initialization seed.')
    Z, logq, proposal = build_proposal(X, num_is, proposal_components, proposal_seed)
    fits, rows = [], []
    for m in grid:
        candidates = [fit_fixed(X, Z, logq, float(m), k, seed=int(s), **fit_options) for s in seeds]
        fit = min(candidates, key=lambda f: f['NLL'])
        fits.append(fit)
        rows.append(dict(m=float(m), NLL=fit['NLL'], converged=fit['converged'],
                         optimizer_failures=fit['optimizer_failures'],
                         post_success=fit['post_success'], post_status=fit['post_status'],
                         post_message=fit['post_message'], post_iterations=fit['post_iterations'],
                         post_attempts=fit['post_attempts'], seed=fit['seed']))
    best = min(fits, key=lambda f: f['NLL'])
    return best, rows, dict(samples=Z, logpdf=logq, model=proposal), fits


def sample_iid(fit, n, seed=123, max_proposals=10_000_000):
    """Independent rejection draws, with a global analytic Gaussian envelope.

    Let a_j = w_j ||x-v_j||^2 and C=k**(m-1). Then
    h(x) >= min_j(a_j)/C, hence exp(-h/sigma^2) <=
    sum_j exp(-w_j ||x-v_j||^2/(sigma^2*C)). Each summand is an
    unnormalized isotropic Gaussian. Select its component proportional to
    its integral and accept with exp(-h/sigma^2)/sum_j exp(-a_j/(sigma^2*C)).
    No fitted normalizing constant, burn-in, or MCMC approximation is needed.
    """
    if n < 1 or int(n) != n:
        raise ValueError('n must be a positive integer.')
    n = int(n)
    rng = np.random.default_rng(seed)
    centers, w = np.asarray(fit['centers']), np.asarray(fit['w'])
    sigma, m = fit['sigma'], fit['m']
    _pack(sigma, centers, w)
    if m <= 1 or not np.isclose(w.sum(), 1):
        raise ValueError('Invalid fitted model.')
    k, d = centers.shape
    logvar = 2*np.log(sigma)+(m-1)*np.log(k)-np.log(w)-np.log(2.)
    logmass = .5*d*logvar
    component_prob = np.exp(logmass-logsumexp(logmass))
    sd = np.exp(.5*logvar)
    chunks, accepted, proposed = [], 0, 0
    while accepted < n:
        count = min(max(256, 2*(n-accepted)), 8192, max_proposals-proposed)
        if count <= 0:
            raise RuntimeError('Rejection sampler proposal budget exceeded; no shortened dataset is returned.')
        labels = rng.choice(k, size=count, p=component_prob)
        Y = centers[labels]+rng.normal(size=(count, d))*sd[labels, None]
        _, h, _ = _terms(Y, centers, np.log(w), m)
        d2 = cdist(Y, centers, 'sqeuclidean')
        log_envelope = logsumexp(-d2/(2*np.exp(logvar)), axis=1)
        log_accept = -h/sigma**2-log_envelope
        if np.max(log_accept) > 1e-8:
            raise RuntimeError('Numerical violation of rejection envelope.')
        keep = Y[np.log(rng.random(count)) <= np.minimum(log_accept, 0)]
        chunks.append(keep)
        accepted += len(keep)
        proposed += count
    return np.concatenate(chunks)[:n]


def align_parameters(centers, w, observed_centers):
    rows, cols = linear_sum_assignment(cdist(observed_centers, centers, 'sqeuclidean'))
    order = np.empty(len(rows), dtype=int)
    order[rows] = cols
    return np.asarray(centers)[order].copy(), np.asarray(w)[order].copy(), order


def bootstrap_summary(estimates, k, d, alpha=.05):
    A = np.asarray(estimates, dtype=float)
    if (A.ndim != 2 or A.shape[1] != 1+(d+1)*k or len(A) < max(d+1, k)
            or not np.isfinite(A).all() or not 0 < alpha < 1 or k < 2):
        raise ValueError('Invalid bootstrap array, B, alpha, or k (requires k >= 2).')
    B = len(A)
    mean, sd = A.mean(axis=0), A.std(axis=0, ddof=0)
    z = norm.ppf(1-alpha/2)
    ci = np.column_stack([mean-z*sd, mean+z*sd])
    ci[0, 0] = max(0., ci[0, 0])  # endpoint 0 denotes intersection with sigma > 0
    ci[1+k*d:, :] = np.clip(ci[1+k*d:, :], 0., 1.)
    center_draws = A[:, 1:1+k*d].reshape(B, k, d)
    covariance = np.stack([np.atleast_2d(np.cov(center_draws[:, j, :], rowvar=False, ddof=1)) for j in range(k)])
    # The manuscript requires an inverse, not a center-covariance pseudoinverse.
    if any(np.linalg.matrix_rank(C) < d for C in covariance):
        raise RuntimeError('Singular center covariance: increase B/check failed or degenerate refits.')
    weight_cov = np.atleast_2d(np.cov(A[:, 1+k*d:], rowvar=False, ddof=1))
    if np.linalg.matrix_rank(weight_cov) < k-1:
        raise RuntimeError('Weight covariance has rank below k-1; increase B/check degeneracy.')
    return dict(B=B, k=k, d=d, alpha=alpha, mean=mean, sd=sd, ci=ci,
                scale_lower_open=bool(ci[0, 0] == 0),
                center_means=mean[1:1+k*d].reshape(k, d),
                center_covariance=covariance, center_precision=np.linalg.inv(covariance),
                center_cutoff=float(chi2.ppf(1-alpha, d)),
                weight_mean=mean[1+k*d:], weight_covariance=weight_cov,
                weight_precision=np.linalg.pinv(weight_cov, hermitian=True),
                weight_cutoff=float(chi2.ppf(1-alpha, k-1)))


def in_center_region(points, summary, center):
    delta = np.asarray(points)-summary['center_means'][center]
    return np.einsum('...i,ij,...j->...', delta, summary['center_precision'][center], delta) <= summary['center_cutoff']


def in_weight_region(weights, summary, atol=1e-10):
    W = np.asarray(weights, dtype=float)
    delta = W-summary['weight_mean']
    quadratic = np.einsum('...i,ij,...j->...', delta, summary['weight_precision'], delta)
    simplex = (W >= 0).all(axis=-1) & (W <= 1).all(axis=-1) & np.isclose(W.sum(axis=-1), 1., atol=atol, rtol=0.)
    return simplex & (quadratic <= summary['weight_cutoff'])


def parametric_bootstrap(X, fit, B=100, alpha=.05, num_is=20000,
                         proposal_components=None, seed=999, fit_options=None):
    """Algorithm inference: iid data, rebuilt proposal/cloud, fixed k,m, alignment."""
    X = _data(X)
    k, d = fit['centers'].shape
    if k < 2 or B < max(d+1, k) or int(B) != B or not 0 < alpha < 1:
        raise ValueError('Require k >= 2, integer B >= max(d+1,k), and 0 < alpha < 1.')
    if (X.shape[1] != d or not fit['converged'] or fit['optimizer_failures']
            or (not fit['post_success'] and fit['post_status'] != 1)):
        raise RuntimeError('Check dimensions and observed blockwise convergence before bootstrapping.')
    options = dict(fit.get('fit_options', {}))
    if fit_options:
        options.update(fit_options)
    rng = np.random.default_rng(seed)
    estimates, diagnostics = [], []
    for b in range(int(B)):
        sample_seed, proposal_seed, refit_seed = map(int, rng.integers(0, 2**31-1, size=3))
        Y = sample_iid(fit, len(X), seed=sample_seed)
        Z, logq, _ = build_proposal(Y, num_is, proposal_components, proposal_seed)
        refit = fit_fixed(Y, Z, logq, fit['m'], k, initial_sigma=fit['sigma'],
                          initial_centers=fit['centers'], initial_w=fit['w'],
                          seed=refit_seed, **options)
        if (not refit['converged'] or refit['optimizer_failures']
                or (not refit['post_success'] and refit['post_status'] != 1)):
            raise RuntimeError(f'Bootstrap replicate {b+1} did not converge cleanly; no partial inference returned.')
        centers, w, order = align_parameters(refit['centers'], refit['w'], fit['centers'])
        estimates.append(np.r_[refit['sigma'], centers.ravel(), w])
        diagnostics.append(dict(replicate=b+1, sample_seed=sample_seed,
                                proposal_seed=proposal_seed, refit_seed=refit_seed,
                                permutation=order.tolist(), NLL=refit['NLL'],
                                post_success=refit['post_success'],
                                IS_diagnostics=refit['IS_diagnostics']))
        print(f'Bootstrap {b+1}/{B}: fixed k={k}, m={fit["m"]:.6g}', flush=True)
    estimates = np.asarray(estimates)
    return dict(estimates=estimates, summary=bootstrap_summary(estimates, k, d, alpha), diagnostics=diagnostics)


from pathlib import Path
import json
import pandas as pd

def report_inference(bootstrap, output_dir):
    summary = bootstrap['summary']
    k, d = summary['k'], summary['d']
    names = ['sigma']+[f'v{a+1}_{j+1}' for a in range(k) for j in range(d)]+[f'w{a+1}' for a in range(k)]
    table = pd.DataFrame({'parameter': names, 'bootstrap_mean': summary['mean'],
                          'bootstrap_sd_ddof0': summary['sd'],
                          'lower': summary['ci'][:, 0], 'upper': summary['ci'][:, 1]})
    display(table)
    if summary['scale_lower_open']:
        print('The scale interval lower endpoint 0 is excluded (sigma > 0).')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    table.to_csv(output_dir/'bootstrap_scalar_intervals.csv', index=False)
    np.savez_compressed(output_dir/'bootstrap_regions.npz', estimates=bootstrap['estimates'],
                        **{key: value for key, value in summary.items()})
    (output_dir/'bootstrap_diagnostics.json').write_text(json.dumps(bootstrap['diagnostics'], indent=2))
    print('Regions saved to', output_dir/'bootstrap_regions.npz')
    print('Use in_center_region(points, summary, center_index) and in_weight_region(weights, summary).')
    return summary, table

# Compatibility for the separate exploratory LR cell; it does not select m.
def em_like_fuzzy_clustering_c(data, proposal_samples, proposal_weights, m, c,
                                initial_sigma=1., initial_centers=None, initial_w=None,
                                max_iter=200, tol=1e-5, random_state=321):
    if np.any(np.asarray(proposal_weights) <= 0):
        raise ValueError('Nonpositive proposal pdf; use logpdf with fit_fixed instead.')
    return fit_fixed(data, proposal_samples, np.log(proposal_weights), m, c,
                     initial_sigma=initial_sigma, initial_centers=initial_centers,
                     initial_w=initial_w, max_outer=max_iter, delta=tol, seed=random_state)

def partial_negative_log_likelihood_c(params, data, U, proposal_samples, proposal_weights, m):
    # U is retained only for compatibility; this is the actual model likelihood.
    d = data.shape[1]
    k = (len(params)-1)//(d+1)
    params = np.asarray(params)
    theta = _pack(params[0], params[1:1+k*d].reshape(k,d), params[1+k*d:])
    return nll_gradient(theta, data, proposal_samples, np.log(proposal_weights), m, k)[0]

def plot_ellipse_2d(ax, mean2d, cov2d, chi2_val=5.991, **kwargs):
    """
    Plots a 2D ellipse corresponding to the contour:
        (x - mean)^T * cov^-1 * (x - mean) = chi2_val
    For a 95% region in 2D, chi2_val ~ 5.991 (df=2).

    mean2d : (2,) array
    cov2d  : (2,2) array
    """
    eigvals, eigvecs = np.linalg.eigh(cov2d)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    angle = np.degrees(np.arctan2(*eigvecs[:,0][::-1]))
    width  = 2.0 * np.sqrt(eigvals[0] * chi2_val)
    height = 2.0 * np.sqrt(eigvals[1] * chi2_val)
    ell = patches.Ellipse(xy=mean2d, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ell)


def normalized_weighted_xbi(X, U, centers, w, m):
    """Manuscript Sec. 4.3: k * sum(w_j*u_ij**m*d_ij**2)/(n*min separation**2)."""
    X, U, centers, w = map(lambda a: np.asarray(a, dtype=float), (X,U,centers,w))
    if X.ndim != 2 or U.shape != (len(X),len(centers)) or centers.shape[1] != X.shape[1]:
        raise ValueError('Incompatible data/membership/center shapes.')
    k=len(centers)
    if k<2 or w.shape!=(k,) or np.any(w<=0) or not np.isclose(w.sum(),1.) or m<=1:
        raise ValueError('Invalid k, weights or fuzzifier.')
    d2=cdist(X,centers,'sqeuclidean');sep=cdist(centers,centers,'sqeuclidean')
    np.fill_diagonal(sep,np.inf);minimum=sep.min()
    if not np.isfinite(minimum) or minimum<=0:
        return np.inf
    return float(k*np.sum(w[None,:]*U**m*d2)/(len(X)*minimum))


def fit_is_clean(fit):
    return bool(fit['converged'] and not fit['optimizer_failures'] and
                (fit['post_success'] or fit['post_status']==1))


def select_k_reference(X, Z, logq, k_grid, reference_m, seed, fit_options,
                       min_hard_count=5, on_degenerate='exclude', output_dir=None):
    """Select k at fixed m; all application-level admissibility decisions are exported.

    Numerical failures stop selection rather than silently removing a k.
    Degenerate reference partitions are either explicitly excluded or stop the run.
    """
    if on_degenerate not in ('exclude','error'):
        raise ValueError('on_degenerate must be exclude or error.')
    if min_hard_count<1 or int(min_hard_count)!=min_hard_count:
        raise ValueError('min_hard_count must be a positive integer.')
    grid=np.asarray(k_grid)
    if grid.ndim!=1 or len(grid)==0 or len(set(grid))!=len(grid) or np.any(grid<2) or np.any(grid!=grid.astype(int)):
        raise ValueError('k_grid must contain distinct integers >=2.')
    rows=[];fits={}
    for candidate_k in grid.astype(int):
        fit=fit_fixed(X,Z,logq,float(reference_m),int(candidate_k),seed=int(seed),**fit_options)
        fits[int(candidate_k)]=fit
        counts=np.bincount(fit['U'].argmax(axis=1),minlength=int(candidate_k))
        value=normalized_weighted_xbi(X,fit['U'],fit['centers'],fit['w'],reference_m)
        clean=fit_is_clean(fit);represented=bool(np.all(counts>=min_hard_count))
        eligible=bool(clean and represented and np.isfinite(value))
        reason=('eligible' if eligible else 'numerical failure' if not clean else
                f'fewer than {min_hard_count} hard assignments in a reference cluster' if not represented else
                'undefined weighted XBI')
        rows.append(dict(k=int(candidate_k),reference_m=float(reference_m),WXBIrel=value,
                         eligible=eligible,reason=reason,min_hard_count=int(counts.min()),
                         hard_counts=json.dumps(counts.tolist()),NLL=fit['NLL'],
                         sigma=fit['sigma'],min_weight=float(fit['w'].min()),
                         blockwise_converged=fit['converged'],blockwise_iterations=fit['n_iter'],
                         blockwise_failures=fit['optimizer_failures'],post_status=fit['post_status'],
                         initialization_seed=int(seed)))
        table=pd.DataFrame(rows)
        if output_dir is not None:
            Path(output_dir).mkdir(parents=True,exist_ok=True)
            table.to_csv(Path(output_dir)/'k_selection_reference.csv',index=False)
        print(f"Reference m={reference_m:g}, k={candidate_k}: WXBIrel={value:.6g}, {reason}",flush=True)
    table=pd.DataFrame(rows)
    if any(not fit_is_clean(fit) for fit in fits.values()):
        raise RuntimeError('A candidate k did not converge; inspect k_selection_reference.csv. No k selected.')
    if on_degenerate=='error' and not table.eligible.all():
        raise RuntimeError('A degenerate reference partition was found. Inspect the exported candidate table.')
    valid=table.loc[table.eligible]
    if valid.empty:
        raise RuntimeError('No eligible reference clustering; no k selected.')
    selected_k=int(valid.loc[valid.WXBIrel.idxmin(),'k'])
    return selected_k,table,fits


def quadrature_nll_2d(fit, X, tail_sd=9., epsabs=1e-9, epsrel=1e-7):
    """Independent endpoint audit; does not refit or replace fitting-cloud selection.

    The integration box contains each Gaussian envelope's +/-tail_sd SD box.
    Outside-box omitted mass is bounded by G*4*Normal.sf(tail_sd).
    Reported numerical integration errors are adaptive-quadrature estimates.
    """
    import math
    from scipy.integrate import quad
    from scipy.stats import norm
    X=np.asarray(X,dtype=float);V=np.asarray(fit['centers'],dtype=float)
    weights=np.asarray(fit['w'],dtype=float);sigma=float(fit['sigma']);m=float(fit['m'])
    if X.ndim!=2 or X.shape[1]!=2 or V.shape[1]!=2:
        raise ValueError('This independent quadrature diagnostic is for two PCs only.')
    if sigma<=0 or m<=1 or np.any(weights<=0):raise ValueError('Invalid fitted density.')
    k=len(weights);variance=sigma**2*k**(m-1)/(2*weights);sd=np.sqrt(variance)
    lower=(V-tail_sd*sd[:,None]).min(axis=0);upper=(V+tail_sd*sd[:,None]).max(axis=0)
    constants=list(zip(V[:,0],V[:,1],np.log(weights)));logsigma2=2*np.log(sigma)
    def kernel(y,x):
        terms=[]
        for cx,cy,lw in constants:
            d2=(x-cx)**2+(y-cy)**2
            if d2==0:return 1.
            terms.append(-(lw+math.log(d2))/(m-1))
        maximum=max(terms);lse=maximum+math.log(sum(math.exp(v-maximum) for v in terms))
        logenergy=-(m-1)*lse-logsigma2
        return 0. if logenergy>700 else math.exp(-math.exp(logenergy))
    inner_errors=[]
    def inner(x):
        value,error=quad(kernel,lower[1],upper[1],args=(x,),points=np.unique(V[:,1]),
                         epsabs=epsabs,epsrel=epsrel,limit=300)
        inner_errors.append(error);return value
    value,outer_error=quad(inner,lower[0],upper[0],points=np.unique(V[:,0]),
                           epsabs=epsabs,epsrel=epsrel,limit=300)
    tail_bound=float(np.sum(2*np.pi*variance)*4*norm.sf(tail_sd))
    error=float(outer_error+max(inner_errors,default=0.)*(upper[0]-lower[0])+tail_bound)
    if not np.isfinite(value) or value<=0:raise RuntimeError('Quadrature normalizer failed.')
    # This direct energy calculation is independent of IS normalizer estimation.
    d2=cdist(X,V,'sqeuclidean')
    logdist=np.log(np.maximum(d2,np.finfo(float).tiny))
    terms=-(logdist+np.log(weights))/(m-1)
    energy=np.exp(-(m-1)*logsumexp(terms,axis=1))/sigma**2
    energy[np.any(d2==0,axis=1)]=0.
    return dict(quadrature_NLL=float(len(X)*np.log(value)+energy.sum()),
                normalizer=float(value),estimated_NLL_quadrature_error=float(len(X)*error/value),
                omitted_mass_upper_bound=tail_bound,tail_sd=float(tail_sd))


In [ ]:
# Two-stage application selection: m=2 -> k by WXBIrel -> m by refined NLL.
K_GRID = np.arange(2, 8)
REFERENCE_M = 2.0
MIN_REFERENCE_HARD_COUNT = 5
DEGENERATE_REFERENCE_POLICY = 'exclude'  # explicit application rule; raw scores are retained
INITIALIZATION_SEED = 321
PROPOSAL_SEED = 674
PROPOSAL_COMPONENTS = 3  # proposal mixture count, distinct from target k
M_GRID = np.geomspace(1.1, 6, 10)
NUM_IS = 20000
K_POST = 1000
FIT_OPTIONS = dict(max_outer=2000, epsilon=1e-5, delta=1e-5,
                   inner_maxiter=1000, K_post=K_POST)
data = pc_scaled2
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Shared observed-data iid cloud for both reference-k and subsequent m fits.
samples_IS, logq_IS, proposal_model = build_proposal(
    data, NUM_IS, PROPOSAL_COMPONENTS, PROPOSAL_SEED)
importance = dict(samples=samples_IS, logpdf=logq_IS, model=proposal_model)
K, k_selection_table, reference_fits = select_k_reference(
    data, samples_IS, logq_IS, K_GRID, REFERENCE_M, INITIALIZATION_SEED,
    FIT_OPTIONS, min_hard_count=MIN_REFERENCE_HARD_COUNT,
    on_degenerate=DEGENERATE_REFERENCE_POLICY, output_dir=OUTPUT_DIR)
display(k_selection_table)
# Diagnostic sensitivity to the declared reference-cluster size rule.
sensitivity=[]
for minimum in [2,5,10]:
    subset=k_selection_table.loc[k_selection_table.min_hard_count.ge(minimum)]
    chosen=int(subset.loc[subset.WXBIrel.idxmin(),'k']) if len(subset) else None
    sensitivity.append(dict(minimum_hard_assignments=minimum,selected_k=chosen))
sensitivity_table=pd.DataFrame(sensitivity)
sensitivity_table.to_csv(OUTPUT_DIR/'k_size_rule_sensitivity.csv',index=False)
display(sensitivity_table)
fig, ax = plt.subplots(figsize=(7, 4.5))
valid = k_selection_table.eligible
ax.plot(k_selection_table.loc[valid, 'k'], k_selection_table.loc[valid, 'WXBIrel'],
        marker='o', label='Eligible reference fits')
excluded=k_selection_table.loc[~valid & np.isfinite(k_selection_table.WXBIrel)]
if len(excluded):
    ax.scatter(excluded.k,excluded.WXBIrel,marker='x',color='gray',s=90,
               label='Excluded by reference-cluster size rule')
ax.set_xticks(K_GRID)
ax.scatter([K], [float(k_selection_table.loc[k_selection_table.k.eq(K), 'WXBIrel'].iloc[0])],
           marker='*', s=180, color='red', label=f'Selected k={K}')
ax.set(xlabel='Number of clusters (k)', ylabel='Normalized weighted XBI',
       title=f'Stage 1: k selection at fixed m={REFERENCE_M:g}')
ax.legend();fig.tight_layout();fig.savefig(OUTPUT_DIR/'k_selection_wxbi.png',dpi=180);plt.show()
# Stage 2 uses the existing fixed-k algorithm on the same integration cloud.
candidate_fits=[];search_rows=[]
for candidate_m in M_GRID:
    fit=fit_fixed(data,samples_IS,logq_IS,float(candidate_m),K,
                  seed=INITIALIZATION_SEED,**FIT_OPTIONS)
    candidate_fits.append(fit)
    block_history=[h for h in fit['history'] if h['stage']=='blockwise']
    search_rows.append(dict(m=float(candidate_m),k=K,NLL=fit['NLL'],
        converged=fit['converged'],optimizer_failures=fit['optimizer_failures'],
        post_success=fit['post_success'],post_status=fit['post_status'],seed=fit['seed'],
        blockwise_iterations=fit['n_iter'],parameter_change=block_history[-1]['parameter_change'],
        NLL_change=block_history[-1]['NLL_change']))
search_table=pd.DataFrame(search_rows)
search_table.to_csv(OUTPUT_DIR/'m_selection_nll.csv',index=False)
display(search_table)
if not all(fit_is_clean(f) for f in candidate_fits):
    raise RuntimeError('An m candidate did not converge cleanly; no final fit selected. Inspect m_selection_nll.csv.')
best_result=min(candidate_fits,key=lambda f:f['NLL'])
best_c,best_m=K,best_result['m']
result_fit=result_c=best_result
proposal_pdf=np.exp(logq_IS)
param_names=['sigma']+[f'v{a+1}_{j+1}' for a in range(K) for j in range(data.shape[1])]+[f'w{a+1}' for a in range(K)]
is_diagnostics_table=pd.DataFrame([dict(m=f['m'],**f['IS_diagnostics']) for f in candidate_fits])
is_diagnostics_table.to_csv(OUTPUT_DIR/'importance_diagnostics.csv',index=False)
display(is_diagnostics_table)
np.savez_compressed(OUTPUT_DIR/'observed_fit.npz',sigma=best_result['sigma'],
                    centers=best_result['centers'],w=best_result['w'],m=best_m,U=best_result['U'],
                    data=data,cohort_indices=ss_id2,cell_types=pca_df2['CellType'].to_numpy(dtype=str))
run_settings=dict(selection='two-stage: reference-m WXBIrel, then fixed-k refined NLL',
    k_grid=list(map(int,K_GRID)),reference_m=REFERENCE_M,selected_k=K,k=K,
    min_reference_hard_count=MIN_REFERENCE_HARD_COUNT,
    degenerate_reference_policy=DEGENERATE_REFERENCE_POLICY,
    m_grid=list(map(float,M_GRID)),selected_m=float(best_m),num_is=NUM_IS,
    proposal=proposal_model.name,proposal_components=PROPOSAL_COMPONENTS,
    integration_draws='conditionally iid',proposal_seed=PROPOSAL_SEED,
    initialization_seed=INITIALIZATION_SEED,initialization_rng='numpy.default_rng',
    cohort_seed=42,pca_random_state=42,input_sha256=__import__('hashlib').sha256(data.tobytes()).hexdigest(),fit_options=FIT_OPTIONS,data_path=str(DATA_PATH),output_dir=str(OUTPUT_DIR),
    NLL=float(best_result['NLL']),fitting_IS_diagnostics=best_result['IS_diagnostics'],
    bootstrap_status='not_run',conditioning='fixed input cohort/features and selected k,m; no selection adjustment')
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings,indent=2))
print('Selected k by normalized weighted XBI:',K,'; selected m by refined NLL:',best_m)

# Independent iid-cloud check: diagnostic only; main selection still uses the shared fit cloud.
validation_Z, validation_logq, _ = build_proposal(data,100000,PROPOSAL_COMPONENTS,675)
validation_rows=[]
for candidate in candidate_fits:
    value,_=nll_gradient(candidate['theta'],data,validation_Z,validation_logq,candidate['m'],K)
    diagnostic=importance_diagnostics(candidate,validation_Z,validation_logq)
    validation_rows.append(dict(m=candidate['m'],fitting_NLL=candidate['NLL'],
                                independent_NLL=value,independent_ESS=diagnostic['ESS']))
validation_table=pd.DataFrame(validation_rows)
validation_table.to_csv(OUTPUT_DIR/'independent_normalizer_check.csv',index=False)
display(validation_table)
if not np.isclose(float(validation_table.loc[validation_table.independent_NLL.idxmin(),'m']),best_m):
    raise RuntimeError('Independent normalizer evaluation changes the selected m; increase the integration budget and refit before inference.')

# Accurate, independent 2D endpoint check of the m ranking. 
quad_rows=[]
for candidate in candidate_fits:
    checked=quadrature_nll_2d(candidate,data)
    quad_rows.append(dict(m=candidate['m'],fitting_cloud_NLL=candidate['NLL'],**checked))
quadrature_table=pd.DataFrame(quad_rows)
quadrature_table.to_csv(OUTPUT_DIR/'quadrature_endpoint_check.csv',index=False)
display(quadrature_table)
if (quadrature_table.estimated_NLL_quadrature_error>.1).any():
    raise RuntimeError('The endpoint quadrature check did not attain adequate accuracy.')
quad_m=float(quadrature_table.loc[quadrature_table.quadrature_NLL.idxmin(),'m'])
if not np.isclose(quad_m,best_m):
    raise RuntimeError('Accurate endpoint NLL changes the selected m. Increase NUM_IS and refit before inference.')
run_settings['quadrature_endpoint_ranking_agrees']=True
run_settings['quadrature_NLL_at_selected_fit']=float(quadrature_table.loc[quadrature_table.m.eq(best_m),'quadrature_NLL'].iloc[0])
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings,indent=2))


In [ ]:
# Algorithm inference: fixed selected k,m; no m-grid search in bootstrap refits.
B, ALPHA = 100, 0.05
BOOTSTRAP_SEED = 999
run_settings.update(B=B, alpha=ALPHA, bootstrap_seed=BOOTSTRAP_SEED, bootstrap_status='running')
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings, indent=2))
bootstrap = parametric_bootstrap(data, best_result, B=B, alpha=ALPHA,
                               num_is=NUM_IS, proposal_components=PROPOSAL_COMPONENTS, seed=BOOTSTRAP_SEED)
inference, interval_table = report_inference(bootstrap, OUTPUT_DIR)
boot_estimates = bootstrap['estimates']
param_means, param_stds = inference['mean'], inference['sd']
ci_lo, ci_hi = inference['ci'][:, 0], inference['ci'][:, 1]
cov_full = np.cov(boot_estimates, rowvar=False, ddof=1)
run_settings.update(B=B, alpha=ALPHA, bootstrap_seed=BOOTSTRAP_SEED, bootstrap_status='complete')
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(run_settings, indent=2))

fig, ax = plt.subplots()
ax.scatter(data[:,0], data[:,1], s=5, alpha=.3)
for a in range(K):
    mean = inference['center_means'][a]
    ax.scatter(*mean, marker='X', label=f'Center {a+1}')
    plot_ellipse_2d(ax, mean, inference['center_covariance'][a],
                    chi2_val=inference['center_cutoff'], fill=False, edgecolor=f'C{a}')
ax.legend()
ax.set_title('Bootstrap center confidence regions')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))  # 1 row, 2 columns

# -------------------------------
# Left: PCA scatter
# -------------------------------
sns.scatterplot(
    data=pca_df2, x='PC1', y='PC2', hue='CellType',
    palette='Dark2', s=100, ax=ax1
)
ax1.set_title('First two PCs of Single-Cell RNA Sequencing Data', fontsize = 18, weight = "bold")
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
ax1.legend(title='Cell types', loc='upper right')

# -------------------------------
# Right: Confidence ellipses
# -------------------------------
center_colors = ['tab:green', 'tab:orange', 'tab:purple']
ax2.scatter(data[:, 0], data[:, 1], s=5, alpha=0.3, label="Data")

for j in range(K):
    mean_j = inference['center_means'][j]
    cov_j = inference['center_covariance'][j]
    color = center_colors[j]

    ax2.scatter(*mean_j, c=color, marker='X', s=100,
                label=f'Center {j+1} mean')
    plot_ellipse_2d(
        ax2, mean_j, cov_j,
        chi2_val=inference['center_cutoff'],
        fill=False, edgecolor=color, linewidth=2
    )

ax2.set_title(f'Bootstrap Confidence Ellipses for {K} Centers',
              fontsize=18, weight="bold")
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
center_colors = ['tab:green', 'tab:orange', 'tab:purple']

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 6, height_ratios=[2, 1.2])

ax1 = fig.add_subplot(gs[0, :3])
ax2 = fig.add_subplot(gs[0, 3:])

# Top left: observed cell types
sns.scatterplot(
    data=pca_df2, x='PC1', y='PC2', hue='CellType',
    palette='Dark2', s=100, ax=ax1
)
ax1.set_title('First two PCs of Single-Cell RNA Sequencing Data',
              fontsize=16, weight='bold')
ax1.set(xlabel='PC1', ylabel='PC2')
ax1.legend(title='Cell types', loc='upper right')

# Top right: full view of confidence ellipses
ax2.scatter(data[:, 0], data[:, 1],
            s=5, alpha=0.3, label='Data')

for j in range(K):
    mean_j = inference['center_means'][j]
    cov_j = inference['center_covariance'][j]
    color = center_colors[j]

    ax2.scatter(*mean_j, color=color, marker='X', s=100,
                label=f'Center {j+1} mean')
    plot_ellipse_2d(
        ax2, mean_j, cov_j,
        chi2_val=inference['center_cutoff'],
        fill=False, edgecolor=color, linewidth=2
    )

ax2.set_title(f'Bootstrap Confidence Ellipses for {K} Centers',
              fontsize=16, weight='bold')
ax2.set(xlabel='Standardized PC1', ylabel='Standardized PC2')
ax2.legend(loc='upper right')

# Bottom: separate zoom for each center (K=3)
for j in range(K):
    ax_zoom = fig.add_subplot(gs[1, 2*j:2*j+2])
    mean_j = inference['center_means'][j]
    cov_j = inference['center_covariance'][j]
    color = center_colors[j]

    ax_zoom.scatter(data[:, 0], data[:, 1],
                    s=8, alpha=0.2, color='steelblue')
    plot_ellipse_2d(
        ax_zoom, mean_j, cov_j,
        chi2_val=inference['center_cutoff'],
        fill=False, edgecolor=color, linewidth=2
    )
    ax_zoom.scatter(*mean_j, color=color, marker='X', s=70,
                    zorder=3)

    # Exact axis-aligned extents of the confidence ellipse, plus padding
    half_widths = np.sqrt(
        inference['center_cutoff'] * np.diag(cov_j)
    )
    half_widths = np.maximum(1.8 * half_widths, 1e-8)

    ax_zoom.set_xlim(mean_j[0] - half_widths[0],
                     mean_j[0] + half_widths[0])
    ax_zoom.set_ylim(mean_j[1] - half_widths[1],
                     mean_j[1] + half_widths[1])
    ax_zoom.set_aspect('equal', adjustable='box')
    ax_zoom.set_title(f'Center {j+1}: Magnified 95% Region', color=color,
                      fontsize=13, weight='bold')
    ax_zoom.set(xlabel='Standardized PC1',
                ylabel='Standardized PC2')
    ax_zoom.ticklabel_format(useOffset=False, style='plain')
    ax_zoom.tick_params(labelsize=9)

fig.tight_layout()
plt.show()

In [ ]:
U = result_fit["U"]
# Step 5: Add memberships to DataFrame
for i in range(K):
    df_scaled2[f"Cluster{i+1}"] = U[:, i]

# Optional: Add true cell types for coloring later
df_scaled2['CellType'] = pca_df2['CellType'].values

# Check the final fitted membership matrix before plotting.
if U.shape != (len(data), K) or not np.allclose(U.sum(axis=1), 1.):
    raise ValueError('Expected n-by-k memberships summing to one per observation.')

# One-to-one annotation using mean memberships within each observed cell type.
# This assignment is descriptive and never changes fitted parameters or column order.
cell_type_names = np.array(sorted(pca_df2['CellType'].unique()))
annotation_scores = np.stack([U[pca_df2['CellType'].to_numpy() == name].mean(axis=0)
                              for name in cell_type_names])
rows, cols = linear_sum_assignment(-annotation_scores)
cluster_names = [f'Cluster {j+1}' for j in range(K)]
for r, j in zip(rows, cols):
    cluster_names[j] = f'Cluster {j+1} ( {cell_type_names[r]})'
display(pd.DataFrame(annotation_scores, index=cell_type_names,
                     columns=[f'Cluster {j+1}' for j in range(K)]))
pd.DataFrame(annotation_scores,index=cell_type_names,columns=[f'Cluster {j+1}' for j in range(K)]).to_csv(OUTPUT_DIR/'postfit_celltype_scores.csv')
pd.DataFrame([dict(cluster_id=j+1,description=cluster_names[j]) for j in range(K)]).to_csv(OUTPUT_DIR/'postfit_cluster_mapping.csv',index=False)

# Preserve all fitted memberships. A 3-vertex display is valid only for k=3.
if K == 3:
    corners=np.eye(3)
    points=U @ corners
    fig=plt.figure(figsize=(8,8));ax=fig.add_subplot(111,projection='3d')
    ax.plot_trisurf(*corners.T,triangles=[[0,1,2]],color='lightgray',alpha=.1)
    ax.scatter(*corners.T,c=['red','green','blue'],s=100)
    ax.scatter(*points.T,c=U,s=35,alpha=.8,edgecolor='k',linewidth=.15)
    ax.set(xlim=(0,1),ylim=(0,1),zlim=(0,1),xlabel=cluster_names[0],
           ylabel=cluster_names[1],zlabel=cluster_names[2],title='Fitted memberships on the 3-vertex simplex')
    ax.set_box_aspect([1,1,1]);ax.view_init(elev=30,azim=45)
    handles=[Line2D([0],[0],marker='o',color='w',markerfacecolor=color,
                    label=name,markersize=9) for name,color in zip(cluster_names,['red','green','blue'])]
    ax.legend(handles=handles,title='Cell-type')
else:
    order=np.argsort(pca_df2['CellType'].to_numpy(),kind='stable')
    fig,ax=plt.subplots(figsize=(max(6,K),6))
    im=ax.imshow(U[order],aspect='auto',cmap='viridis',vmin=0,vmax=1)
    ax.set_xticks(range(K),labels=cluster_names,rotation=45,ha='right')
    ax.set(xlabel='Fitted cluster',ylabel='Cells ordered by reference label for display only',
           title=f'Fitted memberships: selected k={K}')
    fig.colorbar(im,ax=ax,label='Membership')
plt.tight_layout();plt.show()


# WFCM vs GMM

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
from scipy.optimize import linear_sum_assignment
import numpy as np
import matplotlib.pyplot as plt

# === Inputs ===
data = pc_scaled2  # your 2D cell data
U_wfcm = result_fit["U"]
centers_wfcm = result_fit["centers"]

# -------------------------------
# 1. Fit GMM with same k as WFCM
# -------------------------------
c_candidates = [U_wfcm.shape[1]]
best_gmm, best_bic, best_c_gmm = None, np.inf, None

for c_val in c_candidates:
    print(f"\n--- Trying GMM with c = {c_val} ---")
    gmm = GaussianMixture(
        n_components=c_val,
        covariance_type="full",
        random_state=12,
        n_init=5,
        max_iter=300
    )
    gmm.fit(data)
    if not gmm.converged_:
        raise RuntimeError('GMM comparison fit did not converge.')
    bic = gmm.bic(data)
    print(f"BIC for c={c_val}: {bic:.2f}")
    if bic < best_bic:
        best_bic = bic
        best_c_gmm = c_val
        best_gmm = gmm

print("\n===== GMM Final Results =====")
print(f"GMM cluster count fixed to selected WFCM k = {best_c_gmm}")
print(f"Best BIC value = {best_bic:.2f}")
print(f"Means:\n{best_gmm.means_}")
print(f"Weights: {best_gmm.weights_}")

# GMM soft assignments
U_gmm = best_gmm.predict_proba(data)
centers_gmm = best_gmm.means_

# -------------------------------
# 2. Align GMM clusters to WFCM
# -------------------------------
cost_matrix = -U_wfcm.T @ U_gmm
row_ind, col_ind = linear_sum_assignment(cost_matrix)
gmm_order=np.empty(K,dtype=int)
gmm_order[row_ind]=col_ind
U_gmm_aligned = U_gmm[:, gmm_order]
centers_gmm_aligned = centers_gmm[gmm_order, :]

# -------------------------------
# 3. Compare: ARI and Xie–Beni
# -------------------------------
def xie_beni_index(X, U, centers, m=2):
    n, k = U.shape
    dist_sq = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2) ** 2
    num = np.sum((U ** m) * dist_sq)
    denom = n * np.min([
        np.linalg.norm(centers[i] - centers[j]) ** 2
        for i in range(k) for j in range(k) if i != j
    ])
    return num / denom


ari = adjusted_rand_score(np.argmax(U_wfcm, axis=1), np.argmax(U_gmm_aligned, axis=1))
xb_wfcm = xie_beni_index(data, U_wfcm, centers_wfcm, m=best_m)
xb_gmm = xie_beni_index(data, U_gmm_aligned, centers_gmm_aligned, m=best_m)

print(f"\n--- Descriptive comparison; XB uses the same m={best_m:.6g} for both methods ---")
print(f"BIC (GMM): {best_bic:.2f}")
print(f"Xie–Beni (WFCM): {xb_wfcm:.4f}")
print(f"Xie–Beni (GMM): {xb_gmm:.4f}")
print(f"ARI (WFCM vs GMM): {ari:.3f}")

In [ ]:

from matplotlib.colors import ListedColormap

# Define custom discrete colors for 3 clusters
cluster_colors = ListedColormap(plt.get_cmap('tab10')(np.arange(K)%10))  # blue, orange, green

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, U, centers, title in zip(
    axes,
    [U_wfcm, U_gmm_aligned],
    [centers_wfcm, centers_gmm_aligned],
    ["WFCM Memberships", "GMM Memberships"]
):
    cluster_ids = np.argmax(U, axis=1)
    scatter = ax.scatter(
        data[:, 0], data[:, 1],
        c=cluster_ids,
        cmap=cluster_colors, vmin=-.5, vmax=K-.5,
        s=25, alpha=0.8, edgecolor='k', linewidth=0.2
    )
    ax.scatter(
        centers[:, 0], centers[:, 1],
        c='black', s=100, marker='X', label='Centers'
    )
    ax.set_title(title, fontsize=22)
    ax.grid(alpha=0.2)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(15, 5.5), constrained_layout=True
)

# Left: k selection at fixed m=2
kt = k_selection_table.sort_values('k')
eligible = kt['eligible'].astype(bool)

ax1.plot(
    kt.loc[eligible, 'k'],
    kt.loc[eligible, 'WXBIrel'],
    'o-', color='tab:blue',
    label='Weighted XBI'
)

excluded = kt.loc[~eligible & np.isfinite(kt['WXBIrel'])]
ax1.scatter(
    excluded['k'], excluded['WXBIrel'],
    marker='x', color='gray', s=90,
    label='Excluded: small cluster (< 5 cell)'
)

selected_xbi = float(kt.loc[kt['k'].eq(K), 'WXBIrel'].iloc[0])
ax1.scatter(
    K, selected_xbi,
    marker='*', color='red', s=200, zorder=5,
    label=f'Selected k={K}'
)

ax1.set(
    xlabel='Number of clusters (k)',
    ylabel='Normalized weighted XBI'
)
ax1.set_title(
    f'Stage 1: k selection at fixed m={REFERENCE_M:g}',
    fontsize=15, weight='bold'
)
ax1.set_xticks(kt['k'].to_numpy())
ax1.legend(loc='upper left', fontsize=9)


# Right: m selection at selected k
mt = search_table.sort_values('m')
qt = quadrature_table.sort_values('m')

ax2.plot(
    mt['m'], mt['NLL'],
    'o-', color='tab:blue',
    label='Importance-sampling NLL'
)

selected_nll = float(
    mt.loc[np.isclose(mt['m'], best_m), 'NLL'].iloc[0]
)
ax2.scatter(
    best_m, selected_nll,
    marker='*', color='red', s=200, zorder=5,
    label=f'Selected m={best_m:.3g}'
)

ax2.set(xlabel='Fuzzifier (m)', ylabel='NLL')
ax2.set_title(
    f'Stage 2: m selection at selected k={K}',
    fontsize=15, weight='bold'
)
ax2.set_xticks(mt['m'].to_numpy())
ax2.set_xticklabels([f'{m:.3g}' for m in mt['m']])
ax2.legend(loc='upper right', fontsize=9)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(
    OUTPUT_DIR / 'k_and_m_selection.png',
    dpi=300, bbox_inches='tight'
)
plt.show()